## GigaAM CTC / RNNT Fine-tuning

This notebook describes the dataset format and provides end-to-end fine-tuning examples using RNNT loss sub-batching, gradient accumulation, DDP, and encoder freezing.


### Dataset

#### Format

The training pipeline supports a `.tsv` manifest format like the example below.

Each row describes one sample:

* `path` - relative or absolute path to the audio file
* `duration` - audio length in seconds
* `transcription_e2e` - verbatim reference text for the E2E head
* `transcription_raw` - normalized reference text for the raw head (optional)

Example:

```tsv
path	duration	transcription_e2e
audio/train/000000.wav	5.265	Вот только они совсем не радовали, а, напротив, потрясали и ужасали.
audio/train/000001.wav	4.268	Убедившись, что никого нет, он приблизился ко мне и понизил голос.
```

Usage:

```bash
python train.py \
    --train_manifest /path/to/train/manifest.tsv \
    --val_manifest /path/to/val/manifest.tsv \
    ...

python eval.py \
    --eval_manifest /path/to/eval/manifest.tsv \
    ...
```

**Note:** By default, samples in the training and validation sets are filtered by duration to the `[0.1s, 20s]` range. You can change these limits with `min_duration` and `max_duration`.
The examples below fine-tune only the E2E head; the pretrained dual-head examples omit `transcription_raw` from their manifests.


#### Loading data

In [1]:
from utils import load_tonebooks

data_dir = load_tonebooks("data", max_duration=25.0, workers=8)
print(f"E2E manifests: {data_dir}/manifest_train.tsv, {data_dir}/manifest_val.tsv")


E2E manifests: data/manifest_train_e2e.tsv, data/manifest_val_e2e.tsv


### Evaluation

##### Run evaluation for pretrained model

In [2]:
! python -u eval.py \
    --model_name v4_rnnt \
    --eval_manifest data/manifest_val.tsv \
    --disable_tqdm

Loaded 4783 samples
Saved predictions to data/predictions/manifest_val_e2e/v4_rnnt/preds.jsonl
WER e2e: 16.91% (11369/67235 words)
WER raw: 4.71% (3076/65302 words)


##### Check its predictions

In [3]:
import json
from itertools import islice

with open("data/predictions/manifest_val/v4_rnnt/preds.jsonl") as file:
    d = json.loads(next(islice(file, 2, 3)))
    print(f"Sample 2:\nRef:  {d['text']}\nPred: {d['pred_text']}\n")

Sample 2:
Ref:  А Марстен... ну, пожалуй, и Дарвальд по-своему прав.
Pred: А Марстон... Ну, пожалуй, и Дарвольд по-своему прав.



### GigaAM-CTC fine-tuning

#### End-to-end CTC, 4 GPUs


In [4]:
! python -u train.py \
    --train_manifest data/manifest_train.tsv \
    --val_manifest data/manifest_val.tsv \
    --model_name v4_ctc \
    --exp_name tonebooks_ctc \
    --max_epochs 1 \
    --val_check_interval 1.0 \
    --batch_size 64 \
    --eval_batch_size 64 \
    --precision bf16-mixed \
    --devices 4 \
    --lr 8e-5 \
    --val_first_batches 4 \
    --save_top_k 1 \
    --save_weights_only \
    --skip_initial_validation \
    --disable_tqdm

  [val] step=349 epoch=0  val/loss=8.822453  val/wer=0.0771
[epoch 0] time: 251.31 sec
Best: checkpoints/models/tonebooks_ctc/gigaam-v4_ctc-epoch=00-step=000349-val_wer=0.0771.ckpt


##### Evaluate model

*Note:* Eval WER is higher, because for the train / val we exclude audio lengths outside the `[0.1s, 20s]` by default.

In [5]:
import glob

ckpts = glob.glob("checkpoints/models/tonebooks_ctc/gigaam-*.ckpt")
ckpt = min(ckpts, key=lambda x: float(x.split("val_wer=")[-1][: -len(".ckpt")]))
print("evaluating:", ckpt)
!python -u eval.py --checkpoint "{ckpt}" --eval_manifest data/manifest_val.tsv --disable_tqdm

evaluating: checkpoints/models/tonebooks_ctc/gigaam-v4_ctc-epoch=00-step=000349-val_wer=0.0771.ckpt
Loaded 4783 samples
Saved predictions to data/predictions/manifest_val_e2e/tonebooks_ctc/step_000349/preds.jsonl
WER e2e: 8.75% (5884/67235 words)
WER raw: 2.74% (1789/65302 words)


In [6]:
import json

step = ckpt.split("step=")[-1].split("-")[0]
preds_path = glob.glob(f"data/predictions/manifest_val/tonebooks_ctc/step_{step}/preds.jsonl")[0]
with open(preds_path) as file:
    d = json.loads(next(islice(file, 2, 3)))
    print(f"Sample 2:\nRef:  {d['text']}\nPred: {d['pred_text']}\n")

Sample 2:
Ref:  А Марстен... ну, пожалуй, и Дарвальд по-своему прав.
Pred: А Марстен Ну, пожалуй, и Дарвальд по-своему прав.


##### The checkpoint can now be used for `load_model` and subsequent workflows

In [7]:
import torch

import gigaam

finetuned_model = gigaam.load_model(ckpt)
finetuned_model.to_onnx("onnx", dtype=torch.float16)

Exported fine-tuned v4_ctc to onnx.


In [8]:
import csv
from gigaam.onnx_utils import load_onnx, infer_onnx
from utils import compute_wer

sessions, model_cfg = load_onnx("onnx", "v4_ctc")
texts = infer_onnx(
    "data/manifest_val.tsv", model_cfg, sessions, batch_size=64,
    num_workers=4, progress=False,
)["e2e"]
with open("data/manifest_val.tsv", encoding="utf-8", newline="") as stream:
    references = list(csv.DictReader(stream, delimiter="\t"))
results = [{"text": row["transcription_e2e"], "pred_text": text} for row, text in zip(references, texts)]
wer_e2e, wer_raw, e2e_err, e2e_w, raw_err, raw_w = compute_wer(results)
print(f"WER e2e: {wer_e2e:.2f}% ({e2e_err}/{e2e_w} words)\nWER raw: {wer_raw:.2f}% ({raw_err}/{raw_w} words)")


WER e2e: 8.77% (5897/67235 words)
WER raw: 2.75% (1795/65302 words)


##### Compare with checkpoint before tuning

In [9]:
! python -u eval.py \
    --model_name v4_ctc \
    --eval_manifest data/manifest_val.tsv \
    --disable_tqdm

Loaded 4783 samples
Saved predictions to data/predictions/manifest_val_e2e/v4_ctc/preds.jsonl
WER e2e: 17.27% (11609/67235 words)
WER raw: 4.84% (3160/65302 words)


In [10]:
with open("data/predictions/manifest_val/v4_ctc/preds.jsonl") as file:
    d = json.loads(next(islice(file, 2, 3)))
    print(f"Sample 2:\nRef:  {d['text']}\nPred: {d['pred_text']}\n")

Sample 2:
Ref:  А Марстен... ну, пожалуй, и Дарвальд по-своему прав.
Pred: А Марстон. Ну, пожалуй, и Дарвильд по-своему прав.



#### Freeze the encoder


In [11]:
! python -u train.py \
    --train_manifest data/manifest_train.tsv \
    --val_manifest data/manifest_val.tsv \
    --model_name v4_ctc \
    --exp_name tonebooks_ctc_frozen \
    --max_epochs 1 \
    --val_check_interval 1.0 \
    --batch_size 64 \
    --eval_batch_size 64 \
    --precision bf16-mixed \
    --devices 4 \
    --freeze_encoder_epochs -1 \
    --lr 8e-5 \
    --val_first_batches 4 \
    --save_top_k 1 \
    --save_weights_only \
    --skip_initial_validation \
    --disable_tqdm

  [val] step=349 epoch=0  val/loss=20.306622  val/wer=0.1264
[epoch 0] time: 92.82 sec
Best: checkpoints/models/tonebooks_ctc_frozen/gigaam-v4_ctc-epoch=00-step=000349-val_wer=0.1264.ckpt


#### Gradient accumulation


In [12]:
! python -u train.py \
    --train_manifest data/manifest_train.tsv \
    --val_manifest data/manifest_val.tsv \
    --model_name v4_ctc \
    --exp_name tonebooks_ctc_accum \
    --max_epochs 1 \
    --val_check_interval 1.0 \
    --batch_size 32 \
    --accumulate_grad_batches 2 \
    --eval_batch_size 64 \
    --precision bf16-mixed \
    --devices 4 \
    --lr 8e-5 \
    --val_first_batches 4 \
    --save_top_k 1 \
    --save_weights_only \
    --skip_initial_validation \
    --disable_tqdm

  [val] step=349 epoch=0  val/loss=7.797996  val/wer=0.0724
[epoch 0] time: 268.86 sec
Best: checkpoints/models/tonebooks_ctc_accum/gigaam-v4_ctc-epoch=00-step=000349-val_wer=0.0724.ckpt


### GigaAM-RNNT fine-tuning

#### End-to-end RNNT, 4 GPUs

We limit validation batches during training.


In [13]:
! python train.py \
    --model_name v4_rnnt \
    --train_manifest data/manifest_train.tsv \
    --val_manifest data/manifest_val.tsv \
    --exp_name tonebooks_rnnt \
    --max_epochs 1 \
    --val_check_interval 1.0 \
    --batch_size 32 \
    --eval_batch_size 64 \
    --lr 2e-5 \
    --precision bf16-mixed \
    --devices 4 \
    --val_first_batches 4 \
    --save_top_k 1 \
    --save_weights_only \
    --skip_initial_validation \
    --disable_tqdm

  [val] step=697 epoch=0  val/loss=5.063885  val/wer=0.0998
[epoch 0] time: 407.06 sec
Best: checkpoints/models/tonebooks_rnnt/gigaam-v4_rnnt-epoch=00-step=000697-val_wer=0.0998.ckpt


#### RNNT loss sub-batching

RNNT loss materializes a `[batch, audio_seq_len, text_seq_len + 1, vocab_size]` tensor. Sub-batching reduces peak loss memory.


In [14]:
! python train.py \
    --model_name v4_rnnt \
    --train_manifest data/manifest_train.tsv \
    --val_manifest data/manifest_val.tsv \
    --exp_name tonebooks_rnnt_subbatch \
    --max_epochs 1 \
    --val_check_interval 1.0 \
    --batch_size 32 \
    --rnnt_subbatch_size 2 \
    --eval_batch_size 64 \
    --lr 2e-5 \
    --precision bf16-mixed \
    --devices 4 \
    --val_first_batches 4 \
    --save_top_k 1 \
    --save_weights_only \
    --skip_initial_validation \
    --disable_tqdm

  [val] step=697 epoch=0  val/loss=5.059092  val/wer=0.0991
[epoch 0] time: 382.55 sec
Best: checkpoints/models/tonebooks_rnnt_subbatch/gigaam-v4_rnnt-epoch=00-step=000697-val_wer=0.0991.ckpt


#### RNNT loss sub-batching with a frozen encoder

Freeze the encoder and train the E2E decoder and joint modules.


In [15]:
! python train.py \
    --model_name v4_rnnt \
    --train_manifest data/manifest_train.tsv \
    --val_manifest data/manifest_val.tsv \
    --exp_name tonebooks_rnnt_frozen \
    --max_epochs 1 \
    --val_check_interval 1.0 \
    --batch_size 32 \
    --rnnt_subbatch_size 2 \
    --freeze_encoder_epochs -1 \
    --eval_batch_size 64 \
    --lr 2e-4 \
    --precision bf16-mixed \
    --devices 4 \
    --val_first_batches 4 \
    --save_top_k 1 \
    --save_weights_only \
    --skip_initial_validation \
    --disable_tqdm

  [val] step=697 epoch=0  val/loss=5.858390  val/wer=0.1210
[epoch 0] time: 208.79 sec
Best: checkpoints/models/tonebooks_rnnt_frozen/gigaam-v4_rnnt-epoch=00-step=000697-val_wer=0.1210.ckpt


## New language from an SSL backbone

The `multilingual_v2_ssl` backbone contains the preprocessor and encoder, without an ASR head or vocabulary. Below we add a randomly initialized character-wise head and adapt it to Greek using FLEURS.


### Load the dataset

`load_fleurs` writes train, validation, and test manifests. `transcription_e2e` preserves the original case and punctuation; `transcription_raw` contains normalized text.


In [16]:
from utils import load_fleurs

load_fleurs("data_el", lang="el_gr", max_duration=None, workers=8);


data_el/manifest_train.tsv: 3215 samples
data_el/manifest_val.tsv: 271 samples
data_el/manifest_test.tsv: 650 samples


### Fine-tune a CTC head

The character vocabulary is derived from the training E2E labels and saved for reuse. The new head uses a larger learning rate than the encoder.


In [17]:
! python -u train.py \
    --model_name multilingual_v2_ssl \
    --head ctc \
    --train_manifest data_el/manifest_train.tsv \
    --val_manifest data_el/manifest_val.tsv \
    --build_vocab_from_manifest \
    --save_vocab data_el/vocab.json \
    --output_dir ./checkpoints_el \
    --exp_name el_ssl_ctc \
    --batch_size 8 \
    --eval_batch_size 64 \
    --devices 4 \
    --precision bf16-mixed \
    --lr 1e-4 \
    --head_lr_multiplier 10 \
    --warmup_ratio 0.1 \
    --max_epochs 10 \
    --max_duration 30 \
    --val_check_interval 1.0 \
    --save_top_k 1 \
    --save_weights_only \
    --skip_initial_validation \
    --disable_tqdm


  [val] step=101 epoch=0  val/loss=137.025375  val/wer=0.7244
[epoch 0] time: 44.72 sec
  [val] step=202 epoch=1  val/loss=83.085205  val/wer=0.4608
[epoch 1] time: 42.64 sec
  [val] step=303 epoch=2  val/loss=71.800140  val/wer=0.3651
[epoch 2] time: 41.85 sec
  [val] step=404 epoch=3  val/loss=62.500172  val/wer=0.3169
[epoch 3] time: 40.72 sec
  [val] step=505 epoch=4  val/loss=58.882011  val/wer=0.2822
[epoch 4] time: 40.65 sec
  [val] step=606 epoch=5  val/loss=58.030365  val/wer=0.2552
[epoch 5] time: 40.06 sec
  [val] step=707 epoch=6  val/loss=59.846184  val/wer=0.2416
[epoch 6] time: 39.89 sec
  [val] step=808 epoch=7  val/loss=58.124317  val/wer=0.2330
[epoch 7] time: 39.50 sec
  [val] step=909 epoch=8  val/loss=57.477074  val/wer=0.2322
[epoch 8] time: 39.33 sec
  [val] step=1010 epoch=9  val/loss=57.034302  val/wer=0.2327
[epoch 9] time: 40.22 sec
Best: checkpoints_el/models/el_ssl_ctc/gigaam-v2_multilingual_ssl-epoch=08-step=000909-val_wer=0.2322.ckpt


### Fine-tune an RNN-T head

Use the same backbone and E2E training labels with a randomly initialized RNN-T head.


In [18]:
! python -u train.py \
    --model_name multilingual_v2_ssl \
    --head rnnt \
    --train_manifest data_el/manifest_train.tsv \
    --val_manifest data_el/manifest_val.tsv \
    --vocab data_el/vocab.json \
    --output_dir ./checkpoints_el \
    --exp_name el_ssl_rnnt \
    --batch_size 8 \
    --rnnt_subbatch_size 1 \
    --eval_batch_size 64 \
    --devices 4 \
    --precision bf16-mixed \
    --lr 1e-4 \
    --head_lr_multiplier 10 \
    --warmup_ratio 0.1 \
    --max_epochs 10 \
    --max_duration 30 \
    --val_check_interval 1.0 \
    --save_top_k 1 \
    --save_weights_only \
    --skip_initial_validation \
    --disable_tqdm


  [val] step=101 epoch=0  val/loss=355.845642  val/wer=1.0000
[epoch 0] time: 56.97 sec
  [val] step=202 epoch=1  val/loss=68.299767  val/wer=0.5124
[epoch 1] time: 50.89 sec
  [val] step=303 epoch=2  val/loss=37.959641  val/wer=0.3095
[epoch 2] time: 50.51 sec
  [val] step=404 epoch=3  val/loss=30.751299  val/wer=0.2463
[epoch 3] time: 49.63 sec
  [val] step=505 epoch=4  val/loss=28.095200  val/wer=0.2261
[epoch 4] time: 50.12 sec
  [val] step=606 epoch=5  val/loss=27.101812  val/wer=0.2112
[epoch 5] time: 49.04 sec
  [val] step=707 epoch=6  val/loss=26.768906  val/wer=0.2097
[epoch 6] time: 49.00 sec
  [val] step=808 epoch=7  val/loss=26.971991  val/wer=0.1946
[epoch 7] time: 48.83 sec
  [val] step=909 epoch=8  val/loss=27.166435  val/wer=0.1966
[epoch 8] time: 48.65 sec
  [val] step=1010 epoch=9  val/loss=27.112188  val/wer=0.1948
[epoch 9] time: 48.70 sec
Best: checkpoints_el/models/el_ssl_rnnt/gigaam-v2_multilingual_ssl-epoch=07-step=000808-val_wer=0.1946.ckpt


### Evaluate and load the fine-tuned model

In [19]:
import glob

ckpts = glob.glob("checkpoints_el/models/el_ssl_rnnt/*.ckpt")
ckpt = min(ckpts, key=lambda path: float(path.split("val_wer=")[-1][:-len(".ckpt")]))
print("evaluating:", ckpt)
!python -u eval.py --checkpoint "{ckpt}" --eval_manifest data_el/manifest_test.tsv --batch_size 64 --disable_tqdm


evaluating: checkpoints_el/models/el_ssl_rnnt/gigaam-v2_multilingual_ssl-epoch=07-step=000808-val_wer=0.1946.ckpt
Loaded 650 samples
Saved predictions to data_el/predictions/manifest_test/el_ssl_rnnt/step_000808/preds.jsonl
WER e2e: 18.86% (2850/15112 words)
WER raw: 13.53% (2052/15170 words)


In [20]:
import csv
from pathlib import Path
import gigaam

model = gigaam.load_model(ckpt)
with open("data_el/manifest_test.tsv", encoding="utf-8", newline="") as stream:
    sample = next(islice(csv.DictReader(stream, delimiter="\t"), 2, 3))
prediction = model.transcribe(str(Path("data_el") / sample["path"])).text
print(f"Sample 2:\nRef:  {sample['transcription_e2e']}\nPred: {prediction}")


Sample 2:
Ref:  Αν διασχίζετε τη βόρεια Βαλτική θάλασσα τον χειμώνα, ελέγξτε τη θέση της καμπίνας σας, καθώς όταν το σκάφος περνά μέσα από πάγο δημιουργούνται τρομεροί θόρυβοι για εκείνους που επηρεάζονται περισσότερο.
Pred: Αν διασχίδετε τη βόρεια Βαλτική θάλασσα του χειμώνα, ελέξεν τη θέση της καμπίνας σας, καθώς αν το σκάφος περνάμεσα από πάγου δημιουργούνται τρομεροί θόρυβοι για εκείνους που επηρεάζονται περισσότερο.
